In [7]:
# coding: utf-8
"""
recomendacao_cores.py
Gera uma imagem com a foto da pessoa centralizada e as cores da estação
em leque ao redor, no estilo de análise de coloração pessoal.
"""
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import numpy as np
import math
import os

#Paletas por estação
PALETAS = {
    'spring': {
        'nome': 'Tom Quente de Primavera',
        'fundo':   '#FFF8F0',
        'texto':   '#7A3B00',
        'cores': [
            '#F4A460','#FF7F50','#FFD700','#FF6347',
            '#FFA07A','#FFB347','#FFDAB9','#FF8C69',
            '#E9967A','#FFC87C','#FF9966','#F08030',
            '#FFCC44','#FF704D','#FFD580','#F4C430',
            '#E8722A','#FFB347','#FF7733','#FFA040',
        ]
    },
    'fall': {
        'nome': 'Tom Quente de Outono',
        'fundo':   '#FDF5E6',
        'texto':   '#4A2000',
        'cores': [
            '#8B4513','#D2691E','#CD853F','#B8860B',
            '#A0522D','#CC7722','#556B2F','#704214',
            '#8B6914','#C68642','#9B4A1F','#6B3A2A',
            '#A67C52','#7C5C3E','#5C4033','#3E2723',
            '#B5651D','#8D5524','#D4A017','#7A5230',
        ]
    },
    'summer': {
        'nome': 'Tom Fresco de Verão',
        'fundo':   '#F5F5FA',
        'texto':   '#2E4060',
        'cores': [
            '#B0C4DE','#DDA0DD','#E6E6FA','#C0C0C0',
            '#ADD8E6','#D8BFD8','#B09AB0','#A9A9A9',
            '#9BB5C8','#C8A8C8','#8FA8BE','#D4B0D4',
            '#B8A0C8','#A0B8D0','#C0A8B8','#9898B8',
            '#A8B8C8','#C8B0C0','#B0A8C0','#A0A8B8',
        ]
    },
    'winter': {
        'nome': 'Tom Frio de Inverno',
        'fundo':   '#F0F0F8',
        'texto':   '#0A0A2A',
        'cores': [
            '#000080','#DC143C','#4B0082','#FFFFFF',
            '#000000','#008080','#800020','#191970',
            '#0000CD','#C0392B','#6A0DAD','#E8E8E8',
            '#003366','#CC0033','#330066','#F0F0F0',
            '#001F5B','#990000','#2C0054','#D0D0D0',
        ]
    }
}

_ALIAS = {
    'tom quente de primavera(spring)': 'spring',
    'tom quente de outono(fall)':      'fall',
    'tom fresco de verão(summer)':     'summer',
    'tom legal de inverno(winter)':    'winter',
    'spring': 'spring',
    'fall':   'fall',
    'summer': 'summer',
    'winter': 'winter',
}


def _hex_to_rgb(hex_cor: str) -> tuple:
    h = hex_cor.lstrip('#')
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))


def _normalizar_tom(tom: str) -> str:
    chave = _ALIAS.get(tom.strip().lower())
    if chave is None:
        raise ValueError(f"Tom '{tom}' não reconhecido.")
    return chave


def _foto_circular(foto_path: str, diametro: int) -> Image.Image:
    """Abre a foto, redimensiona e aplica máscara circular."""
    foto = Image.open(foto_path).convert("RGBA")

    # Crop central quadrado
    w, h = foto.size
    lado = min(w, h)
    left   = (w - lado) // 2
    top    = (h - lado) // 2
    foto   = foto.crop((left, top, left + lado, top + lado))
    foto   = foto.resize((diametro, diametro), Image.LANCZOS)

    # Máscara circular
    mascara = Image.new("L", (diametro, diametro), 0)
    ImageDraw.Draw(mascara).ellipse([0, 0, diametro, diametro], fill=255)
    foto.putalpha(mascara)
    return foto


def gerar_paleta(tom: str,
                 foto_path: str,
                 caminho_saida: str | None = None) -> str:
    """
    Gera a imagem com leque de cores e foto centralizada.

    Parâmetros
    ----------
    tom        : estação retornada pelo sistema, ex: 'Tom quente de outono(fall)'
    foto_path  : caminho para a foto da pessoa
    caminho_saida : onde salvar o PNG (default: paleta_<estacao>.png na pasta do módulo)

    Retorno
    -------
    Caminho absoluto da imagem salva.
    """
    chave  = _normalizar_tom(tom)
    paleta = PALETAS[chave]
    cores  = paleta['cores']
    n      = len(cores)

    # ── dimensões ────────────────────────────────────────────────────────────
    W = H = 600
    cx = cy = W // 2
    raio_externo = 260
    raio_interno  = 115   # raio do círculo da foto
    borda_branca  = 8

    # ── canvas ───────────────────────────────────────────────────────────────
    bg_rgb = _hex_to_rgb(paleta['fundo'])
    canvas = Image.new("RGBA", (W, H), bg_rgb + (255,))
    draw   = ImageDraw.Draw(canvas)

    # ── leque de cores ───────────────────────────────────────────────────────
    angulo_fatia = 360 / n
    for i, cor in enumerate(cores):
        start = i * angulo_fatia - 90
        end   = start + angulo_fatia + 0.5   # +0.5 evita fenda entre fatias
        bbox  = [cx - raio_externo, cy - raio_externo,
                 cx + raio_externo, cy + raio_externo]
        draw.pieslice(bbox, start=start, end=end, fill=_hex_to_rgb(cor) + (255,))

    # ── borda branca antes da foto ────────────────────────────────────────────
    r_borda = raio_interno + borda_branca
    draw.ellipse(
        [cx - r_borda, cy - r_borda, cx + r_borda, cy + r_borda],
        fill=(255, 255, 255, 255)
    )

    # ── foto circular colada no centro ────────────────────────────────────────
    foto_d = raio_interno * 2
    foto_circ = _foto_circular(foto_path, foto_d)
    pos = (cx - raio_interno, cy - raio_interno)
    canvas.paste(foto_circ, pos, mask=foto_circ)

    # ── label da estação (canto inferior esquerdo) ───────────────────────────
    draw2 = ImageDraw.Draw(canvas)

    label = paleta['nome']
    fonte_size = 22
    try:
        fonte = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
                                   fonte_size)
    except Exception:
        fonte = ImageFont.load_default()

    padding_x, padding_y = 18, 14
    bbox_txt = draw2.textbbox((0, 0), label, font=fonte)
    tw = bbox_txt[2] - bbox_txt[0]
    th = bbox_txt[3] - bbox_txt[1]

    rx0 = 20
    ry0 = H - th - padding_y * 2 - 20
    rx1 = rx0 + tw + padding_x * 2
    ry1 = H - 20

    # fundo semitransparente
    overlay = Image.new("RGBA", canvas.size, (0, 0, 0, 0))
    ov_draw = ImageDraw.Draw(overlay)
    ov_draw.rounded_rectangle([rx0, ry0, rx1, ry1], radius=8,
                               fill=(0, 0, 0, 140))
    canvas = Image.alpha_composite(canvas, overlay)

    draw3 = ImageDraw.Draw(canvas)
    draw3.text((rx0 + padding_x, ry0 + padding_y), label,
               font=fonte, fill=(255, 255, 255, 255))

    # ── converter e salvar ────────────────────────────────────────────────────
    canvas_rgb = canvas.convert("RGB")

    if caminho_saida is None:
        caminho_saida = os.path.join(
            os.path.dirname(os.path.abspath(__file__)),
            f'paleta_{chave}.png'
        )

    canvas_rgb.save(caminho_saida, dpi=(150, 150))
    print(f'Paleta salva em: {caminho_saida}')
    return caminho_saida